# 🔥 Fenix Core — دفتر تدريب عقل Fenix الخاص

**الخطوات:**
1. من شريط الأعلى: **Runtime (بيئة التنفيذ) ← Change runtime type (تغيير نوع البيئة) ← T4 GPU ← Save**
2. اضغط زر ▶ على كل خلية بالترتيب (انتظر كل خلية تخلص)
3. آخر خلية يعطيك **ملف fenix-core-lora.zip** — حمّله وأرسله لي

⏱ الوقت الكلي: 30–60 دقيقة تقريباً

## 1️⃣ التحقق من وجود الـ GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "❌ اختر T4 GPU من: Runtime ← Change runtime type ثم أعد تشغيل هذه الخلية"
print("✅ GPU جاهز:", torch.cuda.get_device_name(0))

## 2️⃣ رفع بيانات التدريب — ارفع ملف `training-data.zip` من جهازك (أو تجاوز الخلية إذا رفعت المجلد يدوياً)

> سيعطيك Fenix ملف `training-data.zip` جاهز — ارفعه في هذه الخلية

In [ ]:
from google.colab import files
up = files.upload()  # اختر training-data.zip
name = list(up.keys())[0]
!unzip -o "{name}" -d /content/
!mkdir -p /content/fenix-core
!cp -r /content/training/data /content/fenix-core/ 2>/dev/null || true
!ls -la /content/fenix-core/data/

## 3️⃣ تثبيت مكتبات التدريب (~دقيقتين)

In [ ]:
%pip install -q -U peft trl bitsandbytes datasets transformers accelerate sentencepiece
import transformers, peft, datasets
print("✅ المكتبات جاهزة — transformers", transformers.__version__)

## 4️⃣ التدريب (الخلية الطويلة ~30-60 دقيقة)

سيرفع النموذج Qwen3-4B ويضيف عليه "طبقات تعلم" صغيرة (LoRA) — دون المساس بالنموذج الأصلي.
ستشاهد `train_loss` ينزل تدريجياً و `eval_loss` يُقاس كل جولة.

In [ ]:
import json, hashlib, time
from pathlib import Path

BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
DATA_DIR = Path("/content/fenix-core/data")
OUT = Path("/content/fenix-core/runs") / time.strftime("%Y%m%d-%H%M%S")
OUT.mkdir(parents=True, exist_ok=True)

def sha1_file(p):
    h = hashlib.sha1()
    with open(p, 'rb') as f:
        for c in iter(lambda: f.read(1<<20), b''): h.update(c)
    return h.hexdigest()[:12]

def load_jsonl(p):
    return [json.loads(l) for l in open(p, encoding='utf-8') if l.strip()]

train_rows = load_jsonl(DATA_DIR / 'train.jsonl')
val_rows = load_jsonl(DATA_DIR / 'val.jsonl')
print(f"train={len(train_rows)}  val={len(val_rows)}  base={BASE_MODEL}")
assert len(train_rows) >= 10, "بيانات قليلة جداً — راجع ملف train.jsonl"

import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
                          DataCollatorForSeq2Seq, EarlyStoppingCallback, Trainer, TrainingArguments)

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
def fmt(rows):
    texts = [tok.apply_chat_template(r['messages'], tokenize=False) for r in rows]
    return Dataset.from_dict({'text': texts})

ds_train, ds_val = fmt(train_rows), fmt(val_rows)
tok_fn = lambda ex: tok(ex['text'], truncation=True, max_length=2048)

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map='auto')
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    task_type='CAUSAL_LM', bias='none'))
model.print_trainable_parameters()

args = TrainingArguments(
    output_dir=str(OUT), num_train_epochs=3,
    per_device_train_batch_size=2, per_device_eval_batch_size=2,
    gradient_accumulation_steps=8, learning_rate=2e-4,
    lr_scheduler_type='cosine', warmup_ratio=0.03, logging_steps=5,
    eval_strategy='epoch', save_strategy='epoch',
    load_best_model_at_end=True, metric_for_best_model='eval_loss',
    greater_is_better=False, bf16=True, optim='paged_adamw_8bit',
    report_to=[], seed=42,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
trainer = Trainer(model=model, args=args,
                  train_dataset=ds_train.map(tok_fn, batched=False, remove_columns=['text']),
                  eval_dataset=ds_val.map(tok_fn, batched=False, remove_columns=['text']),
                  data_collator=DataCollatorForSeq2Seq(tok, padding=True))
result = trainer.train()

manifest = {
    'base_model': BASE_MODEL, 'stamp': OUT.name,
    'train_samples': len(train_rows), 'val_samples': len(val_rows),
    'train_sha1': sha1_file(DATA_DIR / 'train.jsonl'),
    'val_sha1': sha1_file(DATA_DIR / 'val.jsonl'),
    'lora': {'r': 16, 'alpha': 32, 'dropout': 0.05},
    'hyper': {'epochs': 3, 'lr': 2e-4, 'batch': 2, 'accum': 8, 'seed': 42},
    'final_train_loss': getattr(result, 'training_loss', None),
    'best_val_loss': trainer.state.best_metric,
    'best_checkpoint': str(trainer.state.best_model_checkpoint),
}
(OUT / 'run.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(json.dumps(manifest, indent=2))
print('✅ التدريب انتهى — انتقل للخلية التالية')

## 5️⃣ فحص سريع — جرّب عقلك الجديد مباشرة!

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

best = trainer.state.best_model_checkpoint
tok2 = AutoTokenizer.from_pretrained(BASE_MODEL)
m2 = AutoModelForCausalLM.from_pretrained(BASE_MODEL, device_map='auto',
     quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16))
m2 = PeftModel.from_pretrained(m2, best)

def ask(q):
    text = tok2.apply_chat_template(
        [{'role':'system','content':'You are Fenix, an AI assistant built by Hakari.'},
         {'role':'user','content':q}], tokenize=False, add_generation_prompt=True)
    ids = tok2(text, return_tensors='pt').to(m2.device)
    out = m2.generate(**ids, max_new_tokens=250, do_sample=True, temperature=0.6)
    return tok2.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True)

print('🧪 سؤال 1:', ask('Who are you and who built you?'))
print()
print('🧪 سؤال 2:', ask('من أنت بالضبط ومن بنى you؟'))
print()
print('🧪 سؤال 3:', ask('Write a Python function to reverse a string.'))

## 6️⃣ الحفظ — حمّل ملف النموذج المدرّب

In [ ]:
!cd /content/fenix-core && zip -rq fenix-core-lora.zip runs/
!ls -lh /content/fenix-core/fenix-core-lora.zip
from google.colab import files
files.download('/content/fenix-core/fenix-core-lora.zip')
print('📥 حمّل الملف — سنرفعه على خادم Fenix معاً في الخطوة التالية')